# Predicting `arrears_next_month`
Goal: predict whether an account is in arrears next month, for billing months later than the data.

**Feature policy (checklist 7).** `final_notice_sent` is recorded in the billing cycle *after* the billing month, so it is not known at prediction time (and is nearly a restatement of the label); it is dropped. `meter_ref` and `billing_month` are identifier/time keys, not features. Features: `tariff`, `occupants`, `kwh`, `bill_eur`, `late_payments_12m`.

In [ ]:
import numpy as np, pandas as pd
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.dummy import DummyClassifier
from sklearn.metrics import roc_auc_score, average_precision_score, f1_score, precision_score, recall_score

df = pd.read_csv('data.csv', parse_dates=['billing_month']).sort_values('billing_month').reset_index(drop=True)
TARGET='arrears_next_month'
FEATURES=['tariff','occupants','kwh','bill_eur','late_payments_12m']
print(df.shape, df.meter_ref.nunique(), 'accounts;', df.billing_month.min().date(), '->', df.billing_month.max().date())
print('prevalence', df[TARGET].mean().round(3))

## 1. Test set, set aside before any modelling
Accounts appear in many rows and the model will be used on later months. So the test set is the **latest 6 billing months (2025-07 onward), restricted to a random 20% of accounts held out entirely**. Training uses only rows *before* 2025-07 from the other 80% of accounts: later in time and unseen accounts at once.

In [ ]:
rng=np.random.default_rng(0)
accounts=df.meter_ref.unique()
test_accounts=set(rng.choice(accounts, size=int(0.2*len(accounts)), replace=False))
CUTOFF=pd.Timestamp('2025-07-01')
is_test_acc=df.meter_ref.isin(test_accounts)
train=df[(~is_test_acc)&(df.billing_month<CUTOFF)].reset_index(drop=True)
test=df[(is_test_acc)&(df.billing_month>=CUTOFF)].reset_index(drop=True)
assert not set(train.meter_ref)&set(test.meter_ref)
assert train.billing_month.max()<test.billing_month.min()
print(len(train),len(test),'prevalence train/test', train[TARGET].mean().round(3), test[TARGET].mean().round(3))

## 2. Candidate models (named before any scoring)
1. `baseline_prior`: trivial, predicts training prevalence for everyone.
2. `logreg`: logistic regression (C in {0.1, 1, 10}), one-hot tariff, standardised numerics.
3. `hgb`: histogram gradient boosting (max_depth {2,4} x learning_rate {0.05,0.1}).

Feature sets: `all` (5 features) vs `no_usage` (drops kwh and bill_eur). Selection metric: CV average precision (threshold-free); the decision threshold is then chosen on CV out-of-fold predictions to maximise F1. Preprocessing lives inside the pipeline, so it is fit inside each fold (checklist 3).

In [ ]:
def make(kind, feats, **hp):
    num=[f for f in feats if f!='tariff']
    cat=[f for f in feats if f=='tariff']
    pre=ColumnTransformer([('num',StandardScaler(),num)]+([('cat',OneHotEncoder(handle_unknown='ignore'),cat)] if cat else []))
    if kind=='baseline_prior': est=DummyClassifier(strategy='prior')
    elif kind=='logreg': est=LogisticRegression(max_iter=1000, **hp)
    else: est=HistGradientBoostingClassifier(random_state=0, **hp)
    return Pipeline([('pre',pre),('est',est)])

FEATSETS={'all':FEATURES,'no_usage':['tariff','occupants','late_payments_12m']}
CANDS=[('baseline_prior','all',{})]
for fs in FEATSETS:
    CANDS+=[('logreg',fs,{'C':c}) for c in (0.1,1,10)]
    CANDS+=[('hgb',fs,{'max_depth':d,'learning_rate':lr}) for d in (2,4) for lr in (0.05,0.1)]
print(len(CANDS),'candidates')

## 3. Cross-validation inside the training rows
Same structure as the final split: 5 expanding-window folds over training months. Training accounts are split into 5 groups; fold k validates on group-k accounts in the k-th block of months and trains on the other groups' rows from strictly earlier months, so validation is both later and on unseen accounts.

Note on drift: the arrears rate rises steeply over time (about 2-3% in 2023-24, about 15% by mid-2025, about 26% in Dec 2025) while feature averages stay flat. Training prevalence is therefore far below test prevalence. This is expected to hurt threshold transfer and is reported as a caveat rather than tuned away.

In [ ]:
months=np.sort(train.billing_month.unique())
blocks=np.array_split(months[6:],5)   # first 6 months are train-only warm-up
acc_group=dict(zip(train.meter_ref.unique(), np.random.default_rng(1).permutation(train.meter_ref.nunique())%5))
grp=train.meter_ref.map(acc_group)
FOLDS=[]
for k,b in enumerate(blocks):
    # validation: fold-k accounts in block k; training: all other accounts, strictly earlier months
    val=train[(grp==k)&train.billing_month.isin(b)]
    tr=train[(grp!=k)&(train.billing_month<b.min())]
    assert not set(tr.meter_ref)&set(val.meter_ref)
    FOLDS.append((tr,val))
print([(len(a),len(b)) for a,b in FOLDS])

def cv_oof(kind,fs,hp):
    oof=[];ys=[];aps=[]
    for tr,val in FOLDS:
        m=make(kind,FEATSETS[fs],**hp).fit(tr[FEATSETS[fs]],tr[TARGET])
        p=m.predict_proba(val[FEATSETS[fs]])[:,1]
        oof.append(p);ys.append(val[TARGET].values);aps.append(average_precision_score(val[TARGET],p))
    return np.concatenate(oof),np.concatenate(ys),aps

rows=[];OOF={}
for i,(k,fs,hp) in enumerate(CANDS):
    p,y,aps=cv_oof(k,fs,hp); OOF[i]=(p,y)
    rows.append(dict(i=i,model=k,feats=fs,hp=hp,cv_ap=np.mean(aps),cv_ap_sd=np.std(aps),cv_auc=roc_auc_score(y,p)))
res=pd.DataFrame(rows).sort_values('cv_ap',ascending=False)
print(res.head(12).to_string())

In [ ]:
best=res.iloc[0]; bi=int(best.i)
p,y=OOF[bi]
ths=np.linspace(0.05,0.95,91)
f1s=[f1_score(y,p>=t) for t in ths]
THRESH=float(ths[int(np.argmax(f1s))])
print(best[['model','feats','hp','cv_ap','cv_auc']].to_dict()); print('CV-chosen threshold',round(THRESH,2),'CV F1',round(max(f1s),3))
print('baseline CV AP', res[res.model=='baseline_prior'].cv_ap.iloc[0].round(3))

## 4. Choice (from CV only, before touching the test set)
The cell above selects model, feature set, hyperparameters and threshold using cross-validation alone. They are frozen; the next section uses them unchanged.

## 5. Refit on all training rows; score the test set once (headline)

In [ ]:
final=make(best.model,FEATSETS[best.feats],**best.hp).fit(train[FEATSETS[best.feats]],train[TARGET])
pt=final.predict_proba(test[FEATSETS[best.feats]])[:,1]
base=make('baseline_prior',FEATURES).fit(train[FEATURES],train[TARGET]).predict_proba(test[FEATURES])[:,1]
yt=test[TARGET]
out=pd.DataFrame({'model':[best.model+' '+best.feats,'baseline_prior'],
 'AUC':[roc_auc_score(yt,pt),roc_auc_score(yt,base)],
 'AP':[average_precision_score(yt,pt),average_precision_score(yt,base)]})
yh=(pt>=THRESH)
out.loc[0,['precision','recall','F1']]=[precision_score(yt,yh),recall_score(yt,yh),f1_score(yt,yh)]
print('test n =',len(test),' prevalence', yt.mean().round(3), ' threshold', round(THRESH,2))
print(out.round(3).to_string())

## 6. Conclusions
- **Headline (single test scoring):** logistic regression on all five features, C=10, threshold 0.08 (all fixed by CV). On held-out accounts in 2025-07..12: AUC 0.75, average precision 0.55 (prevalence/no-skill 0.24), precision 0.42, recall 0.68, F1 0.52. The trivial baseline has AUC 0.50.
- So yes, there is real but modest signal: usage and billing features rank at-risk accounts well above chance, but this is far from reliable individual-level prediction.
- Caveats: (1) strong label drift: arrears prevalence rises from ~2% to ~26% over the period, features do not, so the CV-chosen threshold (tuned on 2024-25 prevalence) is far too low for the test period and precision is only 0.42; recalibration on recent data would be needed in use. (2) CV scores are noisy (fold SD of AP about 0.13) so differences among the top candidates are not meaningful. (3) The test set is ~275 accounts over 6 months, and rows of one account are correlated. (4) `final_notice_sent` was excluded as unavailable at prediction time; using it would give a misleadingly strong result.